In [8]:
import pandas as pd
import numpy as np

dataset = pd.read_csv("titanic.csv")

print(dataset)

     PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                             Allen, Mr. William Henry    male  35.0      0   
..                                                 ...     ...   ... 

In [9]:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(
    dataset,
    test_size=0.30,
    random_state=42
)

print('train set\n', train_set)

train set
      PassengerId  Survived  Pclass                              Name     Sex  \
445          446         1       1         Dodge, Master. Washington    male   
650          651         0       3                 Mitkoff, Mr. Mito    male   
172          173         1       3      Johnson, Miss. Eleanor Ileen  female   
450          451         0       2             West, Mr. Edwy Arthur    male   
314          315         0       2                Hart, Mr. Benjamin    male   
..           ...       ...     ...                               ...     ...   
106          107         1       3  Salkjelsvik, Miss. Anna Kristine  female   
270          271         0       1             Cairns, Mr. Alexander    male   
860          861         0       3           Hansen, Mr. Claus Peter    male   
435          436         1       1         Carter, Miss. Lucile Polk  female   
102          103         0       1         White, Mr. Richard Frasar    male   

      Age  SibSp  Parch     

In [10]:
features = ['Sex', 'Age', 'Pclass', 'Fare']

train_data = train_set[features].copy()
test_data = test_set[features].copy()

train_data['Sex'] = train_data['Sex'].map({
    'male': 0,
    'female': 1
})

test_data['Sex'] = test_data['Sex'].map({
    'male': 0,
    'female': 1
})

train_data['Age'] = train_data['Age'].fillna(train_data.groupby('Pclass')['Age'].transform('mean'))
test_data['Age'] = test_data['Age'].fillna(test_data.groupby('Pclass')['Age'].transform('mean'))

print('train data\n', train_data)


train data
      Sex        Age  Pclass      Fare
445    0   4.000000       1   81.8583
650    0  25.029098       3    7.8958
172    1   1.000000       3   11.1333
450    0  36.000000       2   27.7500
314    0  43.000000       2   26.2500
..   ...        ...     ...       ...
106    1  21.000000       3    7.6500
270    0  37.893443       1   31.0000
860    0  41.000000       3   14.1083
435    1  14.000000       1  120.0000
102    0  21.000000       1   77.2875

[623 rows x 4 columns]


In [11]:
train_label = train_set['Survived'].copy()
test_label = test_set['Survived'].copy()

print('label train\n', train_label)

label train
 445    1
650    0
172    1
450    0
314    0
      ..
106    1
270    0
860    0
435    1
102    0
Name: Survived, Length: 623, dtype: int64


In [12]:
train_min = train_data.min()
train_max = train_data.max()

# Normalisasi min-max
def min_max_normalization(data, new_min, new_max):
    return (data - train_min) * (new_max - new_min) / (train_max - train_min) + new_min

train_data = min_max_normalization(train_data, 0, 1)
print("\nTrain data setelah normalisasi min-max:\n")
print(train_data)



Train data setelah normalisasi min-max:

     Sex       Age  Pclass      Fare
445  0.0  0.044986     0.0  0.159777
650  0.0  0.309237     1.0  0.015412
172  1.0  0.007288     1.0  0.021731
450  0.0  0.447097     0.5  0.054164
314  0.0  0.535059     0.5  0.051237
..   ...       ...     ...       ...
106  1.0  0.258608     1.0  0.014932
270  0.0  0.470890     0.0  0.060508
860  0.0  0.509927     1.0  0.027538
435  1.0  0.170646     0.0  0.234224
102  0.0  0.258608     0.0  0.150855

[623 rows x 4 columns]


In [13]:
test_data = (test_data - train_min) * (1 - 0) / (train_max - train_min) + 0
print("\nTest data setelah normalisasi min-max:\n")
print(test_data)


Test data setelah normalisasi min-max:

     Sex       Age  Pclass      Fare
709  0.0  0.314212     1.0  0.029758
439  0.0  0.384267     0.5  0.020495
840  0.0  0.246042     1.0  0.015469
720  1.0  0.070118     0.5  0.064412
39   1.0  0.170646     1.0  0.021942
..   ...       ...     ...       ...
821  0.0  0.334004     1.0  0.016908
633  0.0  0.483307     0.0  0.000000
456  0.0  0.811510     0.0  0.051822
500  0.0  0.208344     1.0  0.016908
430  0.0  0.346569     0.0  0.051822

[268 rows x 4 columns]


In [14]:
from sklearn.neighbors import KNeighborsClassifier
# Klasisfikasi k-NN (k = 3) untuk masing-masing pendekatan validasi dan hitung error ratio

knn = KNeighborsClassifier(n_neighbors=3)

# Latih model k-NN dengan data latih
knn.fit(train_data, train_label)

# Prediksi label untuk data uji
prediksi = knn.predict(test_data)

# Hitung error ratio
error = (prediksi != test_label).astype(int)
error_ratio = error.sum() / len(test_label)

print("Jumlah error:", error.sum())
print("Jumlah data :", len(test_label))
print("Error Ratio :", error_ratio)
print("Error (%)   :", error_ratio * 100, "%")

Jumlah error: 50
Jumlah data : 268
Error Ratio : 0.1865671641791045
Error (%)   : 18.65671641791045 %
